# M4 — Evaluation and Analysis

Locating numbers in text images

Assigned member: Shreyansh Kothari (230988)

Prepared supporting notebook. Review and run before adopting its contribution declaration.

The calculations were executed against the supplied project files during report preparation. This portable copy starts with a Colab Drive mount. Adjust PROJECT if the folder has a different name. Install numpy, pandas, matplotlib and pillow if absent.

This notebook recomputes recognition, localization and joint metrics from frozen cached detections. It checks every predicted string and IoU against the saved per-image CSV. It does not rerun the detector. Run M1 first to extract the dataset for the example figures.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

PROJECT = Path(os.environ.get('EE604_PROJECT', '/content/drive/MyDrive/NumericalTextLocalization'))
DATA = Path(os.environ.get('EE604_DATA', '/content/ee604_dataset'))
OUTPUT = Path(os.environ.get('EE604_OUTPUT', str(PROJECT / 'results/M4_evaluation')))
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.environ.get('EE604_CACHE', str(PROJECT / 'results/val_detection_cache.json')))
CSV = Path(os.environ.get('EE604_PREDICTIONS', str(PROJECT / 'results/member2_length_analysis/per_image_predictions.csv')))
TRAIN = Path(os.environ.get('EE604_TRAIN', str(PROJECT / 'experiments/svhn_digit_baseline/results.csv')))
cache = json.loads(CACHE.read_text())

def box_iou(a, b):
    intersection = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersection
    return intersection / union if union > 0 else 0.0

rows = []
for item in cache:
    digits = sorted(item['detections'], key=lambda d: (d['x1'] + d['x2']) / 2)
    text = ''.join(str(d['digit']) for d in digits)
    box = [min(d['x1'] for d in digits), min(d['y1'] for d in digits),
           max(d['x2'] for d in digits), max(d['y2'] for d in digits)] if digits else None
    overlap = box_iou(box, item['gt_box']) if box else 0.0
    exact = text == item['gt_text']
    localized = overlap >= .5
    rows.append({'image': item['filename'], 'gt_text': item['gt_text'], 'predicted_text': text,
                 'length': min(len(item['gt_text']), 5), 'has_prediction': bool(digits),
                 'exact': exact, 'iou': overlap, 'localized': localized, 'joint': exact and localized})
frame = pd.DataFrame(rows)
frame.to_csv(OUTPUT / 'audited_predictions.csv', index=False)
saved = pd.read_csv(CSV, dtype={'image': str, 'gt_text': str, 'predicted_text': str}, keep_default_na=False).set_index('image')
current = frame.set_index('image').loc[saved.index]
assert (current.gt_text == saved.gt_text).all()
assert (current.predicted_text == saved.predicted_text).all()
assert np.allclose(current.iou, saved.iou.astype(float), atol=1e-10)
count = len(frame)
tp = int(frame.localized.sum())
predicted = int(frame.has_prediction.sum())
metrics = {'images': count, 'predicted_sequences': predicted, 'exact_count': int(frame.exact.sum()),
           'localized_count': tp, 'joint_count': int(frame.joint.sum()),
           'exact_accuracy': float(frame.exact.mean()), 'joint_accuracy': float(frame.joint.mean()),
           'localization_precision': tp / predicted, 'localization_recall': tp / count,
           'localization_f1': 2 * tp / (count + predicted), 'mean_iou_all': float(frame.iou.mean()),
           'mean_iou_matched': float(frame.loc[frame.localized, 'iou'].mean()),
           'exact_but_not_localized': int((frame.exact & ~frame.localized).sum()),
           'csv_predictions_verified': True}
(OUTPUT / 'audit_metrics.json').write_text(json.dumps(metrics, indent=2))
lengths = frame.groupby('length').agg(images=('exact', 'size'), exact_count=('exact', 'sum'),
    exact_accuracy=('exact', 'mean'), joint_accuracy=('joint', 'mean'), mean_iou=('iou', 'mean'))
lengths.to_csv(OUTPUT / 'length_analysis.csv')
print(json.dumps(metrics, indent=2))
print(lengths.to_string())

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
x = np.arange(len(lengths))
axes[0].bar(x - .18, lengths.exact_accuracy * 100, .36, label='Exact recognition')
axes[0].bar(x + .18, lengths.joint_accuracy * 100, .36, label='Recognition + IoU >= 0.5')
axes[0].set(xticks=x, xticklabels=['1', '2', '3', '4', '5+'], ylabel='Accuracy (%)', xlabel='Digits per number', ylim=(0, 105))
for pos, n in zip(x, lengths.images): axes[0].text(pos, 99, f'n={n}', ha='center', fontsize=8)
axes[0].legend(fontsize=8, loc='lower center', frameon=False)
train = pd.read_csv(TRAIN)
axes[1].plot(train.epoch, train['metrics/mAP50(B)'] * 100, label='Validation mAP@50')
axes[1].plot(train.epoch, train['metrics/mAP50-95(B)'] * 100, label='Validation mAP@50:95')
axes[1].set(xlabel='Training epoch', ylabel='mAP (%)', ylim=(35, 100))
axes[1].legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT / 'analysis.pdf', bbox_inches='tight')
plt.close(fig)

examples = [3, 4, 36, 54, 1, 82]
lookup = {item['filename']: item for item in cache}
fig, axes = plt.subplots(2, 3, figsize=(9, 4.6))
for ax, index in zip(axes.flat, examples):
    stem = f'val_{index:06d}'
    item = lookup[stem]
    row = frame[frame.image == stem].iloc[0]
    image = Image.open(DATA / 'images/val' / (stem + '.jpg'))
    width, height = image.size
    ax.imshow(image)
    boxes = [(item['gt_box'], '#00ff80', '-', 2)]
    digits = item['detections']
    if digits:
        predicted_box = [min(d['x1'] for d in digits), min(d['y1'] for d in digits), max(d['x2'] for d in digits), max(d['y2'] for d in digits)]
        boxes.append((predicted_box, '#ff4080', '--', 1.8))
    for box, color, style, line_width in boxes:
        ax.add_patch(Rectangle((box[0] * width, box[1] * height), (box[2] - box[0]) * width,
                     (box[3] - box[1]) * height, fill=False, edgecolor=color, linestyle=style, linewidth=line_width))
    ax.set_title(f'{stem}\nGT {row.gt_text} | Pred {row.predicted_text} | IoU {row.iou:.2f}', fontsize=9)
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT / 'failure_examples.png', dpi=220, bbox_inches='tight')
plt.close(fig)
